In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json


from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pickle
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMRegressor


# Métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# ------------------- CONFIGURACIÓN ------------------- #

COLUMNAS_MODELO = [
    "superficie_construida", "banos", "distrito_encoded", "habitaciones",
    "planta_numerica", "exterior", "antiguedad", "terraza", "garaje", "calefaccion"
]



# ------------------- CARGAR Y PREPROCESAR ------------------- #

df = pd.read_csv("sale_properties_clustered.csv")
df = df.dropna()

# Encoding por media del precio
df["distrito_encoded"] = df["distrito"].map(df.groupby("distrito")["price_eur"].mean())

# Imputar y escalar todo el dataset para clustering
imputer = SimpleImputer(strategy="mean")
X_imputado = imputer.fit_transform(df[COLUMNAS_MODELO])

scaler_global = StandardScaler()
X_scaled_global = scaler_global.fit_transform(X_imputado)

# Entrenar KMeans
kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto')
df["cluster"] = kmeans.fit_predict(X_scaled_global)

# Guardar
with open("imputer_sale.pkl", "wb") as f:
    pickle.dump(imputer, f)

with open("scaler_global_sale.pkl", "wb") as f:
    pickle.dump(scaler_global, f)

with open("kmeans_sale.pkl", "wb") as f:
    pickle.dump(kmeans, f)



# ------------------- MODELOS A COMPARAR POR CLÚSTER ------------------- #

modelos_disponibles = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "LightGBM": LGBMRegressor(random_state=42)
}

# Resultados de comparativa
resultados_modelos = []

for cluster_id in df["cluster"].unique():
    df_cluster = df[df["cluster"] == cluster_id]
    X = df_cluster[COLUMNAS_MODELO]
    y = df_cluster["price_eur"].values.reshape(-1, 1)

    # Escalado
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y).ravel()

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

    for nombre_modelo, modelo in modelos_disponibles.items():
        modelo.fit(X_train, y_train)
        y_pred_scaled = modelo.predict(X_test).reshape(-1, 1)
        y_pred = scaler_y.inverse_transform(y_pred_scaled)
        y_test_inv = scaler_y.inverse_transform(y_test.reshape(-1, 1))

        resultados_modelos.append({
            "Cluster": cluster_id,
            "Modelo": nombre_modelo,
            "R2": round(r2_score(y_test_inv, y_pred), 2),
            "MAE": round(mean_absolute_error(y_test_inv, y_pred), 2),
            "MSE": round(mean_squared_error(y_test_inv, y_pred), 2)
        })

# DataFrame con resultados de comparativa
df_resultados_modelos = pd.DataFrame(resultados_modelos)
mejores_modelos_por_cluster = df_resultados_modelos.sort_values("R2", ascending=False).groupby("Cluster").first().reset_index()

# Guardar tabla comparativa
df_resultados_modelos.to_pickle("comparativa_modelos_por_cluster_sale.pkl")



# ------------------- ENTRENAR MEJOR MODELO POR CLÚSTER ------------------- #

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

mejores_parametros_por_cluster = {}

for cluster_id in df["cluster"].unique():
    print(f"\n🔍 Hiperparámetros para clúster {cluster_id}...")

    df_cluster = df[df["cluster"] == cluster_id]
    X = df_cluster[COLUMNAS_MODELO]
    y = df_cluster["price_eur"].values.reshape(-1, 1)

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y).ravel()

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

    grid_search = GridSearchCV(
        estimator=RandomForestRegressor(random_state=42),
        param_grid=param_grid,
        scoring='r2',
        cv=5,
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    mejores_parametros_por_cluster[cluster_id] = {
        "mejores_parametros": grid_search.best_params_,
        "mejor_score_cv": grid_search.best_score_
    }

    with open(f"mejor_modelo_rf_cluster_{cluster_id}_sale.pkl", "wb") as f:
        pickle.dump(grid_search.best_estimator_, f)
    with open(f"scaler_X_{cluster_id}_sale.pkl", "wb") as f:
        pickle.dump(scaler_X, f)
    with open(f"scaler_y_{cluster_id}_sale.pkl", "wb") as f:
        pickle.dump(scaler_y, f)

# Guardar resultados
pd.DataFrame.from_dict(mejores_parametros_por_cluster, orient='index').to_pickle("mejores_parametros_rf_por_cluster_sale.pkl")



# ------------------- FUNCIÓN DE PREDICCIÓN ------------------- #

def predecir_precio_vivienda(nueva_vivienda: dict) -> float:
    df_nueva = pd.DataFrame([nueva_vivienda])
    df_nueva = df_nueva.reindex(columns=COLUMNAS_MODELO)

    with open("imputer_sale.pkl", "rb") as f:
        imputer = pickle.load(f)
    with open("scaler_global_sale.pkl", "rb") as f:
        scaler_global = pickle.load(f)
    with open("kmeans_sale.pkl", "rb") as f:
        kmeans = pickle.load(f)

    X_imputado = imputer.transform(df_nueva)
    X_scaled_global = scaler_global.transform(X_imputado)
    cluster = kmeans.predict(X_scaled_global)[0]
    print(f"🏷️ Clúster asignado: {cluster}")

    with open(f"mejor_modelo_rf_cluster_{cluster}_sale.pkl", "rb") as f:
        modelo = pickle.load(f)
    with open(f"scaler_X_{cluster}_sale.pkl", "rb") as f:
        scaler_X = pickle.load(f)
    with open(f"scaler_y_{cluster}_sale.pkl", "rb") as f:
        scaler_y = pickle.load(f)

    X_scaled = scaler_X.transform(X_imputado)
    y_pred_scaled = modelo.predict(X_scaled).reshape(-1, 1)
    y_pred = scaler_y.inverse_transform(y_pred_scaled).ravel()[0]

    print(f"✅ Precio estimado: {y_pred:,.2f} EUR")
    return y_pred

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000034 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 237
[LightGBM] [Info] Number of data points in the train set: 247, number of used features: 10
[LightGBM] [Info] Start training from score -0.001894
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -

c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000089 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 311
[LightGBM] [Info] Number of data points in the train set: 648, number of used features: 8
[LightGBM] [Info] Start training from score 0.006676
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000144 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 310
[LightGBM] [Info] Number of data points in the train set: 384, number of used features: 9
[LightGBM] [Info] Start training from score 0.009231
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000111 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 278
[LightGBM] [Info] Number of data points in the train set: 498, number of used features: 9
[LightGBM] [Info] Start training from score 0.039120
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000065 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 449
[LightGBM] [Info] Number of data points in the train set: 623, number of used features: 7
[LightGBM] [Info] Start training from score -0.026521
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



🔍 Hiperparámetros para clúster 2...
Fitting 5 folds for each of 162 candidates, totalling 810 fits

🔍 Hiperparámetros para clúster 4...
Fitting 5 folds for each of 162 candidates, totalling 810 fits

🔍 Hiperparámetros para clúster 1...
Fitting 5 folds for each of 162 candidates, totalling 810 fits

🔍 Hiperparámetros para clúster 3...
Fitting 5 folds for each of 162 candidates, totalling 810 fits


In [2]:
# Comprobación del modelo

import pandas as pd
import numpy as np
import pickle

# Datos nuevos
nueva_vivienda = {
    "superficie_construida": 120,
    "banos": 2,
    "distrito_encoded": 1800000,
    "habitaciones": 3,
    "planta_numerica": 2,
    "exterior": 1,
    "antiguedad": 15,
    "terraza": 1,
    "garaje": 1,
    "calefaccion": 1
}

expected_cols = [
    "superficie_construida", "banos", "distrito_encoded", "habitaciones",
    "planta_numerica", "exterior", "antiguedad", "terraza", "garaje", "calefaccion"
]

# Paso 1: Convertir y reindexar
df_nueva = pd.DataFrame([nueva_vivienda])
df_nueva = df_nueva.reindex(columns=expected_cols)

# Paso 2: Cargar transformadores globales
with open("imputer_sale.pkl", "rb") as f:
    imputer = pickle.load(f)
with open("scaler_global_sale.pkl", "rb") as f:
    scaler_global = pickle.load(f)
with open("kmeans_sale.pkl", "rb") as f:
    kmeans = pickle.load(f)

# Paso 3: Imputar y escalar
X_imputado = imputer.transform(df_nueva)
X_scaled_global = scaler_global.transform(X_imputado)

# Paso 4: Asignar clúster
cluster = kmeans.predict(X_scaled_global)[0]
print(f"🏷️ Clúster asignado: {cluster}")

# Paso 5: Cargar modelo y scalers del clúster
with open(f"mejor_modelo_rf_cluster_{cluster}_sale.pkl", "rb") as f:
    modelo = pickle.load(f)
with open(f"scaler_X_{cluster}_sale.pkl", "rb") as f:
    scaler_X = pickle.load(f)
with open(f"scaler_y_{cluster}_sale.pkl", "rb") as f:
    scaler_y = pickle.load(f)

# Paso 6: Escalar con scaler del clúster
X_scaled_cluster = scaler_X.transform(X_imputado)

# Paso 7: Predecir y revertir escala
y_pred_scaled = modelo.predict(X_scaled_cluster).reshape(-1, 1)
precio_final = scaler_y.inverse_transform(y_pred_scaled).ravel()[0]

print(f"✅ Precio estimado: {precio_final:,.2f} EUR")


🏷️ Clúster asignado: 4
✅ Precio estimado: 1,079,093.15 EUR


c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [3]:
df_resultados_modelos = pd.read_pickle("comparativa_modelos_por_cluster_sale.pkl")
df_resultados_modelos

,Cluster,Modelo,R2,MAE,MSE
0,0,Linear Regression,0.22,1542760.69,5.782753e+12
1,0,Decision Tree,0.43,1173241.94,4.275460e+12
2,0,Random Forest,0.59,953391.52,3.024090e+12
3,0,KNN,0.27,1444545.16,5.423403e+12
4,0,SVR,0.47,1095022.82,3.942846e+12
5,0,LightGBM,0.51,1072988.96,3.610075e+12
6,2,Linear Regression,0.80,255652.08,1.329017e+11
7,2,Decision Tree,0.67,228727.80,2.179156e+11
8,2,Random Forest,0.87,171873.40,8.558660e+10
9,2,KNN,0.84,191226.64,1.059009e+11


In [4]:
df_mejores_parametros_rf = pd.read_pickle("mejores_parametros_rf_por_cluster_sale.pkl")
df_mejores_parametros_rf



,mejores_parametros,mejor_score_cv
0,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.401270
2,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.799446
4,"{'max_depth': 10, 'max_features': 'sqrt', 'min...",0.700009
1,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.827094
3,"{'max_depth': None, 'max_features': 'sqrt', 'm...",0.744756
